# Quads design (3dp pla + blue shims) for static cloaking


## Imports

In [ ]:
import os
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=8"  # Use 8 CPU cores for JAX pmap 

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
import jax.numpy as jnp
import numpy as np
from pathlib import Path
import cv2
from mechanicalmetamaterialcloaks.geometry import QuadGeometry, CloakQuadGeometry
from problems.quads_static_cloaking import ForwardInput, ForwardProblem, OptimizationProblem
from mechanicalmetamaterialcloaks.utils import save_data, load_data, SolutionData
from mechanicalmetamaterialcloaks.plotting import generate_animation, generate_frames, plot_geometry, generate_several_animations, plot_geometry_field_overlaid
from typing import Optional

import jax
jax.config.update("jax_enable_x64", True)  # enable float64 type

plt.style.use(["science", "grid"])
%matplotlib widget

## Plotting functions


In [ ]:
def plot_objective_iterations(optimization: OptimizationProblem, optimization_filename: Optional[str] = None):

    fig, axes = plt.subplots(nrows=3, figsize=(10, 7), sharex=True, constrained_layout=True)
    axes[0].set(ylabel="Objective")
    axes[0].plot(optimization.objective_values, lw=3, color="#2980b9")
    axes[1].set(ylabel="Angle constraints violation")
    axes[1].plot(optimization.constraints_violation["angles"], lw=3, color="#c0392b")
    axes[1].axhline(y=0, color="black")
    axes[2].set(ylabel="Edge length constraints violation")
    axes[2].plot(optimization.constraints_violation["edge_lengths"], lw=3, color="#c0392b")
    axes[2].axhline(y=0, color="black")
    axes[-1].set(xlabel="Iteration")

    if optimization_filename is not None:
        path = Path(
            f"../out/{optimization.name}/{optimization_filename}/objective_iterations.png")
        path.parent.mkdir(parents=True, exist_ok=True)  # Make sure parents directories exist
        fig.savefig(str(path), dpi=300)


# DELTA DIFFERENCE FUNCTIONS
def difference_displacement_field_centroids(solutionData_cg: SolutionData, solutionData_mg: SolutionData, cloaked_geometry: CloakQuadGeometry):
    dimension_less = jnp.array([1/cloaked_geometry.spacing, 1/cloaked_geometry.spacing, 1])
    return ((((solutionData_cg.fields[-1, 0, cloaked_geometry.id_surronding_area_in_the_cloaked_geometry, :]
               - solutionData_mg.fields[-1, 0, cloaked_geometry.id_surronding_area_in_the_mother_geometry, :])*dimension_less)**2).sum()**0.5) \
        / ((solutionData_mg.fields[-1, 0, cloaked_geometry.id_surronding_area_in_the_mother_geometry, :]*dimension_less)**2).sum()**0.5


def difference_ux_centroids(solutionData_cg: SolutionData, solutionData_mg: SolutionData, cloaked_geometry: CloakQuadGeometry):
    return ((((solutionData_cg.fields[-1, 0, cloaked_geometry.id_surronding_area_in_the_cloaked_geometry, 0]
               - solutionData_mg.fields[-1, 0, cloaked_geometry.id_surronding_area_in_the_mother_geometry, 0]))**2).sum()**0.5) \
        / ((solutionData_mg.fields[-1, 0, cloaked_geometry.id_surronding_area_in_the_mother_geometry, 0])**2).sum()**0.5


def difference_uy_centroids(solutionData_cg: SolutionData, solutionData_mg: SolutionData, cloaked_geometry: CloakQuadGeometry):
    return ((((solutionData_cg.fields[-1, 0, cloaked_geometry.id_surronding_area_in_the_cloaked_geometry, 1]
               - solutionData_mg.fields[-1, 0, cloaked_geometry.id_surronding_area_in_the_mother_geometry, 1]))**2).sum()**0.5) \
        / ((solutionData_mg.fields[-1, 0, cloaked_geometry.id_surronding_area_in_the_mother_geometry, 1])**2).sum()**0.5


def difference_theta_centroids(solutionData_cg: SolutionData, solutionData_mg: SolutionData, cloaked_geometry: CloakQuadGeometry):
    return ((((solutionData_cg.fields[-1, 0, cloaked_geometry.id_surronding_area_in_the_cloaked_geometry, 2]
               - solutionData_mg.fields[-1, 0, cloaked_geometry.id_surronding_area_in_the_mother_geometry, 2]))**2).sum()**0.5) \
        / ((solutionData_mg.fields[-1, 0, cloaked_geometry.id_surronding_area_in_the_mother_geometry, 2])**2).sum()**0.5


def difference_ux_centroids_dimension_less(solutionData_cg: SolutionData, solutionData_mg: SolutionData, cloaked_geometry: CloakQuadGeometry, optimization: OptimizationProblem):
    return ((((solutionData_cg.fields[-1, 0, cloaked_geometry.id_surronding_area_in_the_cloaked_geometry, 0]
               - solutionData_mg.fields[-1, 0, cloaked_geometry.id_surronding_area_in_the_mother_geometry, 0])/cloaked_geometry.spacing)**2).sum()) \
        / ((solutionData_mg.fields[-1, 0, cloaked_geometry.id_surronding_area_in_the_mother_geometry, :]*optimization.dimension_less)**2).sum()


def difference_uy_centroids_dimension_less(solutionData_cg: SolutionData, solutionData_mg: SolutionData, cloaked_geometry: CloakQuadGeometry, optimization: OptimizationProblem):
    return ((((solutionData_cg.fields[-1, 0, cloaked_geometry.id_surronding_area_in_the_cloaked_geometry, 1]
               - solutionData_mg.fields[-1, 0, cloaked_geometry.id_surronding_area_in_the_mother_geometry, 1])/cloaked_geometry.spacing)**2).sum()) \
        / ((solutionData_mg.fields[-1, 0, cloaked_geometry.id_surronding_area_in_the_mother_geometry, :]*optimization.dimension_less)**2).sum()


def difference_utheta_centroids_dimension_less(solutionData_cg: SolutionData, solutionData_mg: SolutionData, cloaked_geometry: CloakQuadGeometry, optimization: OptimizationProblem):
    return ((((solutionData_cg.fields[-1, 0, cloaked_geometry.id_surronding_area_in_the_cloaked_geometry, 2]
               - solutionData_mg.fields[-1, 0, cloaked_geometry.id_surronding_area_in_the_mother_geometry, 2]))**2).sum()) \
        / ((solutionData_mg.fields[-1, 0, cloaked_geometry.id_surronding_area_in_the_mother_geometry, :]*optimization.dimension_less)**2).sum()


# GENERATE DELTA DIFFERENCE EVOLUTION
def generate_delta_difference(solutionData_cg: SolutionData, solutionData_mg: SolutionData, cloaked_geometry: CloakQuadGeometry, title: str):
    _initial_delta = difference_displacement_field_centroids(solutionData_cg, solutionData_mg, cloaked_geometry)
    # plot
    fig, ax = plt.subplots()
    geometry = ['initial guess']
    delta_value = [_initial_delta]
    bar_labels = ['red']
    bar_colors = ['tab:red']
    ax.bar(geometry, delta_value, label=bar_labels, color=bar_colors)
    ax.set_ylabel('Delta value')
    ax.set_title(title)
    plt.show()


def generate_delta_difference_optimized_and_initial_geometry_and_save(initial_solutionData_cg: SolutionData, solutionData_mg: SolutionData, optimized_solutionData_cg: SolutionData, cloaked_geometry: CloakQuadGeometry, title: str, out_filename: str):
    fig, ax = plt.subplots()
    _initial_delta = difference_displacement_field_centroids(initial_solutionData_cg, solutionData_mg, cloaked_geometry)
    _optimized_delta = difference_displacement_field_centroids(
        optimized_solutionData_cg, solutionData_mg, cloaked_geometry)

    # plot
    geometry = ['initial guess', 'optimized response']
    delta_value = [_initial_delta, _optimized_delta]
    bar_labels = ['red', 'blue']
    bar_colors = ['tab:red', 'tab:blue']
    ax.bar(geometry, delta_value, label=bar_labels, color=bar_colors)
    ax.set_ylabel('Delta value')
    ax.set_title(title)
    plt.savefig(out_filename+'.jpg', bbox_inches='tight', dpi=150)


def generate_delta_uk_difference_optimized_and_initial_geometry(uk: str, initial_solutionData_cg: SolutionData, solutionData_mg: SolutionData, optimized_solutionData_cg: SolutionData, cloaked_geometry: CloakQuadGeometry):
    if uk == "ux":
        _initial_delta = difference_ux_centroids(initial_solutionData_cg, solutionData_mg, cloaked_geometry)
        _optimized_delta = difference_ux_centroids(optimized_solutionData_cg, solutionData_mg, cloaked_geometry)
        title = 'ux delta difference - opimized geometry and initial geometry'

    if uk == "uy":
        _initial_delta = difference_uy_centroids(initial_solutionData_cg, solutionData_mg, cloaked_geometry)
        _optimized_delta = difference_uy_centroids(optimized_solutionData_cg, solutionData_mg, cloaked_geometry)
        title = 'uy delta difference - opimized geometry and initial geometry'

    if uk == "theta":
        _initial_delta = difference_theta_centroids(initial_solutionData_cg, solutionData_mg, cloaked_geometry)
        _optimized_delta = difference_theta_centroids(optimized_solutionData_cg, solutionData_mg, cloaked_geometry)
        title = 'theta delta difference - opimized geometry and initial geometry'
    if uk == "ux" or uk == "uy" or uk == "theta":
        fig, ax = plt.subplots()
        ax.plot(initial_solutionData_cg.timepoints, _initial_delta, c='blue', label='initial geometry')
        ax.plot(optimized_solutionData_cg.timepoints, _optimized_delta, c='red', label='optimized geometry')
        ax.set(ylabel='difference (dimension less)', xlabel='time (s)', title=title)
        ax.legend()
        plt.show()


def generate_delta_difference_square_uks_contribution(solutionData_cg: SolutionData, solutionData_mg: SolutionData, cloaked_geometry: CloakQuadGeometry, optimization: OptimizationProblem, suffix_title: str = ''):
    contribution_ux_delta_square = difference_ux_centroids_dimension_less(
        solutionData_cg, solutionData_mg, cloaked_geometry, optimization)
    contribution_uy_delta_square = difference_uy_centroids_dimension_less(
        solutionData_cg, solutionData_mg, cloaked_geometry, optimization)
    contribution_utheta_delta_square = difference_utheta_centroids_dimension_less(
        solutionData_cg, solutionData_mg, cloaked_geometry, optimization)
    fig, ax = plt.subplots()
    ax.plot(solutionData_cg.timepoints, contribution_ux_delta_square, c='blue', label='ux')
    ax.plot(solutionData_cg.timepoints, contribution_uy_delta_square, c='red', label='uy')
    ax.plot(solutionData_cg.timepoints, contribution_utheta_delta_square, c='green', label='utheta')
    ax.set(ylabel='difference squared (dimension less)', xlabel='time (s)',
           title='Contribution of ux, uy, utheta in delta squared' + suffix_title)
    ax.legend()
    plt.show()

# PLOT THE DISPLACEMENT OF TARGETED CENTROIDS IN THE MAP


def plot_displacement_1centroid_for_the_three_different_geometries(id_centroid_in_cg: int, id_centroids_in_mg: int, solution_data_cg_optimized_geometry: SolutionData, solution_data_cg_initial_guess: SolutionData, solution_data_mg: SolutionData, title: str = "", t_min: int = 0, t_max: int = None):
    if t_max == None:
        t_max = solution_data_cg_optimized_geometry.timepoints.shape[0]
    fig, axes = plt.subplots(nrows=3, figsize=(10, 10), sharex=True, constrained_layout=True)
    axes[0].set(ylabel="u_x displacement")
    axes[0].plot(solution_data_cg_optimized_geometry.timepoints[t_min:t_max],
                 solution_data_cg_optimized_geometry.fields[t_min:t_max, 0, id_centroid_in_cg, 0], lw=3, color='red', label="optimized geometry")
    axes[0].plot(solution_data_cg_initial_guess.timepoints[t_min:t_max], solution_data_cg_initial_guess.fields[t_min:t_max,
                 0, id_centroid_in_cg, 0], lw=3, color='blue', label="initial guess geometry")
    axes[0].plot(solution_data_mg.timepoints[t_min:t_max], solution_data_mg.fields[t_min:t_max,
                 0, id_centroids_in_mg, 0], lw=3, color='green', label="mother geometry")
    axes[0].legend()

    axes[1].set(ylabel="u_y displacement")
    axes[1].plot(solution_data_cg_optimized_geometry.timepoints[t_min:t_max],
                 solution_data_cg_optimized_geometry.fields[t_min:t_max, 0, id_centroid_in_cg, 1], lw=3, color='red', label="optimized geometry")
    axes[1].plot(solution_data_cg_initial_guess.timepoints[t_min:t_max], solution_data_cg_initial_guess.fields[t_min:t_max,
                 0, id_centroid_in_cg, 1], lw=3, color='blue', label="initial guess geometry")
    axes[1].plot(solution_data_mg.timepoints[t_min:t_max], solution_data_mg.fields[t_min:t_max,
                 0, id_centroids_in_mg, 1], lw=3, color='green', label="mother geometry")
    axes[1].legend()

    axes[2].set(ylabel="theta")
    axes[2].plot(solution_data_cg_optimized_geometry.timepoints[t_min:t_max],
                 solution_data_cg_optimized_geometry.fields[t_min:t_max, 0, id_centroid_in_cg, 2], lw=3, color='red', label="optimized geometry")
    axes[2].plot(solution_data_cg_initial_guess.timepoints[t_min:t_max], solution_data_cg_initial_guess.fields[t_min:t_max,
                 0, id_centroid_in_cg, 2], lw=3, color='blue', label="initial guess geometry")
    axes[2].plot(solution_data_mg.timepoints[t_min:t_max], solution_data_mg.fields[t_min:t_max,
                 0, id_centroids_in_mg, 2], lw=3, color='green', label="mother geometry")
    axes[2].legend()

    axes[0].set_title(title)
    axes[-1].set(xlabel="times (s)")


def plot_displacement_1centroid(id_centroid: int, solution_data: SolutionData):

    fig, axes = plt.subplots(nrows=3, figsize=(10, 10), sharex=True, constrained_layout=True)
    axes[0].set(ylabel="u_x displacement")
    axes[0].plot(solution_data.timepoints, solution_data.fields[:, 0, id_centroid, 0], lw=3, color='red')

    axes[1].set(ylabel="u_y displacement")
    axes[1].plot(solution_data.timepoints, solution_data.fields[:, 0, id_centroid, 1], lw=3, color='red')

    axes[2].set(ylabel="theta")
    axes[2].plot(solution_data.timepoints, solution_data.fields[:, 0, id_centroid, 2], lw=3, color='red')

    axes[-1].set(xlabel="times (s)")


def plot_displacement_1centroid_for_mg_and_initial_guess_cg(id_centroid_in_cg: int, id_centroids_in_mg: int, solution_data_cg_initial_guess: SolutionData, solution_data_mg: SolutionData, title: str = "", t_min: int = 0, t_max: int = None):
    if t_max == None:
        t_max = solution_data_cg_initial_guess.timepoints.shape[0]
    fig, axes = plt.subplots(nrows=3, figsize=(10, 10), sharex=True, constrained_layout=True)
    axes[0].set(ylabel="u_x displacement")
    axes[0].plot(solution_data_cg_initial_guess.timepoints[t_min:t_max], solution_data_cg_initial_guess.fields[t_min:t_max,
                 0, id_centroid_in_cg, 0], lw=3, color='blue', label="geometry with initial guess")
    axes[0].plot(solution_data_mg.timepoints[t_min:t_max],
                 solution_data_mg.fields[t_min:t_max, 0, id_centroids_in_mg, 0], lw=3, color='green')
    axes[0].legend()

    axes[1].set(ylabel="u_y displacement")
    axes[1].plot(solution_data_cg_initial_guess.timepoints[t_min:t_max], solution_data_cg_initial_guess.fields[t_min:t_max,
                 0, id_centroid_in_cg, 1], lw=3, color='blue', label="geometry initial guess")
    axes[1].plot(solution_data_mg.timepoints[t_min:t_max],
                 solution_data_mg.fields[t_min:t_max, 0, id_centroids_in_mg, 1], lw=3, color='green')
    axes[1].legend()

    axes[2].set(ylabel="theta")
    axes[2].plot(solution_data_cg_initial_guess.timepoints[t_min:t_max], solution_data_cg_initial_guess.fields[t_min:t_max,
                 0, id_centroid_in_cg, 2], lw=3, color='blue', label="geometry initial guess")
    axes[2].plot(solution_data_mg.timepoints[t_min:t_max],
                 solution_data_mg.fields[t_min:t_max, 0, id_centroids_in_mg, 2], lw=3, color='green')
    axes[2].legend()

    axes[0].set_title(title)
    axes[-1].set(xlabel="times (s)")


def plot_displacement_several_centroids_for_mg(ids_centroids_in_mg: list, solution_data_mg: SolutionData, title: str = "", t_min: int = 0, t_max: int = None):
    if t_max == None:
        t_max = solution_data_mg.timepoints.shape[0]
    fig, axes = plt.subplots(nrows=3, figsize=(10, 10), sharex=True, constrained_layout=True)
    axes[0].set(ylabel="u_x displacement")
    for k in range(len(ids_centroids_in_mg)):
        axes[0].plot(solution_data_mg.timepoints[t_min:t_max], solution_data_mg.fields[t_min:t_max,
                     0, ids_centroids_in_mg[k], 0], lw=3, label=f"column {k}")
    axes[0].legend()

    axes[1].set(ylabel="u_y displacement")
    for k in range(len(ids_centroids_in_mg)):
        axes[1].plot(solution_data_mg.timepoints[t_min:t_max], solution_data_mg.fields[t_min:t_max,
                     0, ids_centroids_in_mg[k], 1], lw=3, label=f"column {k}")
    # axes[1].legend()

    axes[2].set(ylabel="theta")
    for k in range(len(ids_centroids_in_mg)):
        axes[2].plot(solution_data_mg.timepoints[t_min:t_max], solution_data_mg.fields[t_min:t_max,
                     0, ids_centroids_in_mg[k], 2], lw=3, label=f"column {k}")
    # axes[2].legend()

    axes[0].set_title(title)
    axes[-1].set(xlabel="times (s)")

# GAUGE THE PART OF THE DELTA FUNCTION DUE TO PHASE SHIFT, AND THE PART ONLY DUE TO THE DISCREPANCY OF THE AMPLITUDE


def gauge_contribution_phase_shift_in_delta_function(solutionData_mg: SolutionData, solutionData_cg: SolutionData, cloaked_geometry: CloakQuadGeometry, phase_shift_threshold: int, k_cycle_begin_static_state: int, nb_cycles: int):
    _delta_difference = difference_displacement_field_centroids(solutionData_cg, solutionData_mg, cloaked_geometry)

# ESTIMATE THE FRACTION OF THE EFFECT OF ux, uy and theta IN THE VALUE OF THE DELTA FUCTION INTEGRATED OVER TIME


def fraction_uk_filds_contained_in_delta_objective_value(solutionData_init_guess_cg: SolutionData, solutionData_op_cg: SolutionData, solutionData_mg: SolutionData, optimization: OptimizationProblem, integration_delay: bool = True):
    # retrieve the objective values
    ob_value_initial_guess_cg = optimization.objective_values[0]
    ob_value_optimised_cg = optimization.objective_values[-1]
    if integration_delay == True:
        t_min = 200*optimization.forward_problem.n_cycle_min
    # estimate the fraction of uk fields of the op cg
    ux_op_cg = \
        (
            (
                (solutionData_op_cg.fields[t_min:, 0, optimization.forward_problem.cloaked_geometry.id_surronding_area_in_the_cloaked_geometry, 0]
                 -
                 solutionData_mg.fields[t_min:, 0,
                                        optimization.forward_problem.cloaked_geometry.id_surronding_area_in_the_mother_geometry, 0]
                 )
                / optimization.forward_problem.spacing
            )**2
        ).sum() \
        / (
            jnp.max(((solutionData_mg.fields[t_min:, 0, optimization.forward_problem.cloaked_geometry.id_surronding_area_in_the_mother_geometry, :]
                      * optimization.dimension_less)**2).sum(axis=(1, 2))
                    )
            * ob_value_optimised_cg**2
        )
    uy_op_cg = (((solutionData_op_cg.fields[t_min:, 0, optimization.forward_problem.cloaked_geometry.id_surronding_area_in_the_cloaked_geometry, 1] - solutionData_mg.fields[t_min:, 0, optimization.forward_problem.cloaked_geometry.id_surronding_area_in_the_mother_geometry, 1])/optimization.forward_problem.spacing)**2).sum() \
        / (jnp.max(((solutionData_mg.fields[t_min:, 0, optimization.forward_problem.cloaked_geometry.id_surronding_area_in_the_mother_geometry, :]*optimization.dimension_less)**2).sum(axis=(1, 2))) * ob_value_optimised_cg**2)
    utheta_op_cg = ((solutionData_op_cg.fields[t_min:, 0, optimization.forward_problem.cloaked_geometry.id_surronding_area_in_the_cloaked_geometry, 2] - solutionData_mg.fields[t_min:, 0, optimization.forward_problem.cloaked_geometry.id_surronding_area_in_the_mother_geometry, 2])**2).sum() \
        / (jnp.max(((solutionData_mg.fields[t_min:, 0, optimization.forward_problem.cloaked_geometry.id_surronding_area_in_the_mother_geometry, :]*optimization.dimension_less)**2).sum(axis=(1, 2))) * ob_value_optimised_cg**2)

    # estimate the fraction of uk fields of the initial guess cg
    ux_ig_cg = (((solutionData_init_guess_cg.fields[t_min:, 0, optimization.forward_problem.cloaked_geometry.id_surronding_area_in_the_cloaked_geometry, 0] - solutionData_mg.fields[t_min:, 0, optimization.forward_problem.cloaked_geometry.id_surronding_area_in_the_mother_geometry, 0])/optimization.forward_problem.spacing)**2).sum() \
        / (jnp.max(((solutionData_mg.fields[t_min:, 0, optimization.forward_problem.cloaked_geometry.id_surronding_area_in_the_mother_geometry, :]*optimization.dimension_less)**2).sum(axis=(1, 2))) * ob_value_initial_guess_cg**2)
    uy_ig_cg = (((solutionData_init_guess_cg.fields[t_min:, 0, optimization.forward_problem.cloaked_geometry.id_surronding_area_in_the_cloaked_geometry, 1] - solutionData_mg.fields[t_min:, 0, optimization.forward_problem.cloaked_geometry.id_surronding_area_in_the_mother_geometry, 1])/optimization.forward_problem.spacing)**2).sum() \
        / (jnp.max(((solutionData_mg.fields[t_min:, 0, optimization.forward_problem.cloaked_geometry.id_surronding_area_in_the_mother_geometry, :]*optimization.dimension_less)**2).sum(axis=(1, 2))) * ob_value_initial_guess_cg**2)
    utheta_ig_cg = ((solutionData_init_guess_cg.fields[t_min:, 0, optimization.forward_problem.cloaked_geometry.id_surronding_area_in_the_cloaked_geometry, 2] - solutionData_mg.fields[t_min:, 0, optimization.forward_problem.cloaked_geometry.id_surronding_area_in_the_mother_geometry, 2])**2).sum() \
        / (jnp.max(((solutionData_mg.fields[t_min:, 0, optimization.forward_problem.cloaked_geometry.id_surronding_area_in_the_mother_geometry, :]*optimization.dimension_less)**2).sum(axis=(1, 2))) * ob_value_initial_guess_cg**2)

    # plot
    species = ("optimized cloaked geometry", "initial guess")
    penguin_means = {
        'ux': (ux_op_cg, ux_ig_cg),
        'uy': (uy_op_cg, uy_ig_cg),
        'utheta': (utheta_op_cg, utheta_ig_cg),
    }

    x = np.arange(len(species))  # the label locations
    width = 0.25  # the width of the bars
    multiplier = 0

    fig, ax = plt.subplots(layout='constrained')

    for attribute, measurement in penguin_means.items():
        offset = width * multiplier
        rects = ax.bar(x + offset, measurement, width, label=attribute)
        ax.bar_label(rects, padding=3)
        multiplier += 1

    # Add some text for labels, title and custom x-axis tick labels, etc.
    ax.set_ylabel('fraction (dim less)')
    ax.set_title('Fraction of ux, uy, utheta in the objective value')
    ax.set_xticks(x + width, species)
    ax.legend(loc='upper left')  # , ncols=3)
    ax.set_ylim(0, 1)
    plt.show()

def extract_outline_points_from_image(image_path, threshold=100, n_points=100):
    img = cv2.imread(image_path)
    # Flip y axis to have origin at bottom left
    img = cv2.flip(img, 0)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY_INV)
    # row by row finds first contour (top to bottom), then goes CCW
    cnts, _ = cv2.findContours(
        thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    # target_shape = np.array(cnts[0]).reshape((len(cnts[0]), 2))
    target_shape = cnts[0].reshape((len(cnts[0]), 2))
    M = cv2.moments(target_shape)
    # x centroid: target shape is going to be aligned with this point
    cX = int(M["m10"] / M["m00"])
    idx = jnp.where((target_shape[:, 0] == cX) & (
        target_shape[:, 1] == target_shape[:, 1].min()))
    # shift pixels to "centroid" location
    target_shape = jnp.roll(target_shape, -idx[0], axis=0)

    # downsampling the target pixels to the number of blocks
    idxs = jnp.round(jnp.linspace(0, len(target_shape),
                     n_points, endpoint=False)).astype(int)
    target_shape_reduced = target_shape[idxs, :]
    return target_shape_reduced


def rescale_point_cloud(points, center=(0, 0), width=1., height=1.):
    x, y = points[:, 0], points[:, 1]
    x = (x - x.min()) / (x.max() - x.min()) * width + center[0] - width / 2
    y = (y - y.min()) / (y.max() - y.min()) * height + center[1] - height / 2
    return jnp.array([x, y]).T

## Problem setup


In [ ]:
optimization_name = "quads_static_cloaking_3dp_pla_shims"

# NOTE: Units are mm, N, s

# Geometrical params
n1_blocks = 30
n2_blocks = 30
spacing = 15.  # 1.0  # 15 mm
hinge_length = 0.15*spacing  # Same as bond length
# horizontal_shifts = jnp.zeros((n1_blocks+1, n2_blocks, 2))  # Initial design
# vertical_shifts = jnp.zeros((n1_blocks, n2_blocks+1, 2))  # Initial design
initial_angle = 20*jnp.pi/180
mother_geometry = QuadGeometry(n1_blocks, n2_blocks, spacing=spacing, bond_length=hinge_length)
initial_horizontal_shift, initial_vertical_shift = mother_geometry.get_design_from_rotated_square(initial_angle)
block_centroids_mg, centroid_node_vectors_mg, bond_connectivity_mg, reference_bond_vectors_mg = mother_geometry.get_parametrization()

_init_block_centroids_mg = block_centroids_mg(initial_horizontal_shift, initial_vertical_shift)

# Width strip and void
width_strip_cloak_area = 35.*2.5*2
# Create a snake-like void
image_points = extract_outline_points_from_image(f"../data/inclusion-images/cat.jpg", threshold=100, n_points=100)
image_points = rescale_point_cloud(image_points, center=(0, 0), width=1, height=1)
x0, y0 = spacing*(n1_blocks-1) / 2 + 0.5*spacing, spacing*(n2_blocks-1) / 2 + 0.25*spacing
void_width = 8.*spacing*2
void_height = 8.*spacing*2
void = rescale_point_cloud(image_points, center=(x0+0.25*spacing, y0), width=void_width, height=void_height)
middle_cloak_area = jnp.array([x0, y0])
cloaked_geometry = CloakQuadGeometry(mother_geometry, _init_block_centroids_mg, void,
                                     width_strip_cloak_area=width_strip_cloak_area, middle_cloak_area=middle_cloak_area)
block_centroids, centroid_node_vectors, bond_connectivity, reference_bond_vectors = cloaked_geometry.get_parametrization()

# Initial guess
guessed_horizontal_shift, guessed_vertical_shift = initial_horizontal_shift[
    cloaked_geometry.mask_horizontal_shift_allowed], initial_vertical_shift[cloaked_geometry.mask_vertical_shift_allowed]

# Mechanical params
k_stretch = 120.  # stretching stiffness 120. N/mm
k_shear = 1.19  # shearing stiffness 1.19 N/mm
k_rot = 1.50  # rotational stiffness 1.50 Nmm
density = 1. # 6.18e-9  # Mg/mm^2 # NOTE: This is scaled to 1. just for static problems
damping_scaling = 1.
damping = 0.0186 * jnp.array([
    2 * (0.36125 * density * spacing**2 * k_shear)**0.5,
    2 * (0.36125 * density * spacing**2 * k_shear)**0.5,
    2 * (0.02175026 * density * spacing**4 * k_rot)**0.5
]) * damping_scaling

# Forward input for the two problems to be optimized
amplitude = 2.*spacing
n_timepoints = 200
simulation_time = 7000. # s # NOTE: Scaled up for static problems
# Hz loading frequency for dynamic input
forward_input = ForwardInput(
    # amplitude=amplitude,  # mm
    # loading_rate=loading_rate,  # Hz
    horizontal_shifts=guessed_horizontal_shift,
    vertical_shifts=guessed_vertical_shift,
)

# Forward problem
problem = ForwardProblem(
    n1_blocks=n1_blocks,
    n2_blocks=n2_blocks,
    spacing=spacing,
    bond_length=hinge_length,
    void=void,
    middle_cloak_area=middle_cloak_area,
    width_strip_cloak_area=width_strip_cloak_area,
    horizontal_vertical_shifts_mg=None,
    initial_angle=initial_angle,
    k_stretch=k_stretch,
    k_shear=k_shear,
    k_rot=k_rot,
    density=density,
    damping=damping,
    k_contact=k_rot,
    min_angle=-15*jnp.pi/180,
    cutoff_angle=-10*jnp.pi/180,
    amplitude=amplitude,  # mm
    simulation_time=simulation_time,
    n_timepoints=n_timepoints,
    name=optimization_name
)

# Optimization
optimization = OptimizationProblem(
    forward_problem=problem,
    forward_input=forward_input,
    objective_type="integrated",
    name=optimization_name,
)
problem_filename_prefix = f"quads{'_linearized_strains' if optimization.forward_problem.linearized_strains else ''}_{optimization.forward_problem.n1_blocks}x{optimization.forward_problem.n2_blocks}_amplitude_{optimization.forward_problem.amplitude}_initial_angle_{initial_angle*180/jnp.pi:.1f}"
optimization_filename = f"opt_{optimization.objective_type}_with_angle_30_and_length_3_constraints_{problem_filename_prefix}_void_cat"

# Setup forward problem
problem.setup()
problem.plot_sketch()

### Intact geometry


In [ ]:
horizontal_shifts_init, vertical_shifts_init = mother_geometry.get_design_from_rotated_square(angle=initial_angle)
xlim, ylim = mother_geometry.get_xy_limits(
    horizontal_shifts_init, vertical_shifts_init) + 0.5*mother_geometry.spacing * jnp.array([-1, 1])

solutionData_mg = problem.solutionData_mg
generate_animation(
    solutionData_mg,
    field="u",
    deformed=True,
    out_filename=f"../out/{optimization.name}/{optimization_filename}/{problem_filename_prefix}_intact",
    xlim=xlim,
    ylim=ylim,
    cmap="inferno",
    fps=30,
    dpi=300,
    frame_range=range(0, len(solutionData_mg.timepoints), 2),
    figsize=(6, 5)
)


### Holed geometry


In [ ]:
design_value = guessed_horizontal_shift, guessed_vertical_shift

horizontal_shifts_init, vertical_shifts_init = mother_geometry.get_design_from_rotated_square(angle=initial_angle)
xlim, ylim = mother_geometry.get_xy_limits(
    horizontal_shifts_init, vertical_shifts_init) + 0.5*mother_geometry.spacing * jnp.array([-1, 1])

solution_data_cg_initial_guess = problem.solve(design_value)
generate_animation(
    solution_data_cg_initial_guess,
    field="u",
    deformed=True,
    out_filename=f"../out/{optimization.name}/{optimization_filename}/{problem_filename_prefix}_holed",
    xlim=xlim,
    ylim=ylim,
    cmap="inferno",
    fps=30,
    dpi=300,
    frame_range=range(0, len(solution_data_cg_initial_guess.timepoints), 2),
    figsize=(6, 5)
)


### Disturbance (error: holed - intact)


In [ ]:
generate_delta_difference(solution_data_cg_initial_guess, problem.solutionData_mg, problem.cloaked_geometry,
                          title="Disturbance (holed - intact)")


## Optimization

### Import most recent optimization object


In [ ]:
optimization = OptimizationProblem.from_dict(
    load_data(
        f"../data/{optimization.name}/{optimization_filename}.pkl",
    )
)

### Run optimization

In [ ]:
# optimization.run_optimization_nlopt(
#     initial_guess=(optimization.forward_input.horizontal_shifts,
#                    optimization.forward_input.vertical_shifts),
#     # initial_guess=optimization.design_values[-1],
#     n_iterations=30,
#     min_block_angle=30*jnp.pi/180,
#     min_void_angle=0*jnp.pi/180,
#     min_edge_length=3.,  # mm
#     max_time=12*60*60,  # 12 hours
# )

# save_data(
#     f"../data/{optimization.name}/{optimization_filename}.pkl",
#     optimization.to_dict()  # Optimization problem
# )


## Plots

### Import most recent optimization object


In [ ]:
optimization = OptimizationProblem.from_dict(
    load_data(
        f"../data/{optimization.name}/{optimization_filename}.pkl",
    )
)


### Objective iterations


In [ ]:
plot_objective_iterations(
    optimization=optimization,
    optimization_filename=optimization_filename,
)

### Plot designs


In [ ]:
for solution_data, label in zip(optimization.forward_problem.solution_data, ["intact", "holed", "cloaked"]):
    fig, axes = plot_geometry(
        block_centroids=solution_data.block_centroids,
        centroid_node_vectors=solution_data.centroid_node_vectors,
        bond_connectivity=solution_data.bond_connectivity,
        figsize=(4, 4),
    )
    axes.axis("off")
    fig.savefig(f"../out/{optimization.name}/{optimization_filename}/{label}_geometry.png",
                dpi=300, transparent=True)
    plt.close(fig)


### Snapshots

In [ ]:
cmap = matplotlib.colors.LinearSegmentedColormap.from_list(
    name="custom_cmap",
    colors=[
        "#ff7b00",  # negative
        "#e46e00",  # negative
        "#c86100",  # negative
        "#b25600",  # negative
        "#000000",  # 0
        "#0098b0",  # positive
        "#00aac4",  # positive
        "#00c1df",  # positive
        "#00ddff",  # positive
    ],
)
plt.close("all")
fig, axes = plt.subplots(ncols=3, figsize=(3*3, 2.75), sharex=True, constrained_layout=True)
vlim = (
    min(data.fields[:, 0, :, 1].min() for data in optimization.forward_problem.solution_data),
    max(data.fields[:, 0, :, 1].max() for data in optimization.forward_problem.solution_data),
)

for solution_data, ax in zip(optimization.forward_problem.solution_data, axes):
    plot_geometry_field_overlaid(
        data=solution_data,
        field="uy",
        timepoint=-1,
        deformed=True,
        cmap=cmap,
        vlim=vlim,
        colorbar=False,
        axis=False,
        ax=ax,
    )
    ax.axis("off")
    ax.set_aspect("equal")

# Add colorbar to the last axis
cb = fig.colorbar(
    matplotlib.cm.ScalarMappable(cmap=cmap, norm=matplotlib.colors.Normalize(vmin=vlim[0], vmax=vlim[1])),
    ax=axes[-1],
    orientation='vertical',
    pad=0.08,
    aspect=15,
)
cb.ax.tick_params(labelsize=16)
cb.set_label(r"$u_y$ [mm]", fontsize=18)
fig.savefig(f"../out/{optimization.name}/{optimization_filename}/uy_displacement_reference_holed_cloaked.png",
            dpi=300, bbox_inches='tight')

### Response animation: Optimized cloaked geometry

In [ ]:
horizontal_shifts_init, vertical_shifts_init = mother_geometry.get_design_from_rotated_square(angle=initial_angle)
xlim, ylim = mother_geometry.get_xy_limits(
    horizontal_shifts_init, vertical_shifts_init) + 0.5*mother_geometry.spacing * jnp.array([-1, 1])

solution_data_cg_optimized_geometry = optimization.forward_problem.solution_data[-1]
generate_animation(
    solution_data_cg_optimized_geometry,
    field="u",
    deformed=True,
    out_filename=f"../out/{optimization.name}/{optimization_filename}/{problem_filename_prefix}_cloaked",
    xlim=xlim,
    ylim=ylim,
    cmap="inferno",
    fps=30,
    dpi=300,
    frame_range=range(0, len(solution_data_cg_optimized_geometry.timepoints), 2),
    figsize=(6, 5)
)


### Response animation: Intact vs holed vs cloaked geometry


In [ ]:
cmap = matplotlib.colors.LinearSegmentedColormap.from_list(
    name="custom_cmap",
    colors=[
        "#ff7b00",  # negative
        "#e46e00",  # negative
        "#c86100",  # negative
        "#b25600",  # negative
        "#000000",  # 0
        "#0098b0",  # positive
        "#00aac4",  # positive
        "#00c1df",  # positive
        "#00ddff",  # positive
    ],
)

horizontal_shifts_init, vertical_shifts_init = mother_geometry.get_design_from_rotated_square(angle=initial_angle)
xlim, ylim = mother_geometry.get_xy_limits(
    horizontal_shifts_init, vertical_shifts_init) + 0.5*mother_geometry.spacing * jnp.array([-1, 1])

generate_several_animations(
    # [solutionData_mg, solution_data_cg_initial_guess, solution_data_cg_optimized_geometry]
    optimization.forward_problem.solution_data,
    field='uy',
    out_filename=f"../out/{optimization.name}/{optimization_filename}/{problem_filename_prefix}_intact_holed_cloaked_comparison_uy_custom_cmap",
    row_or_column='row',
    xlim=xlim,
    ylim=ylim,
    fps=30,
    dpi=300,
    cmap=cmap,
    vlim=(-30, 30),
    frame_range=range(0, len(optimization.forward_problem.solution_data[0].timepoints), 2),
    figsize=(12.5, 4),
    # legend_label="Displacement [mm]",
    legend_label="Displacement $u_y$ [mm]",
    axis=False,
)
